# Chatbot PetShop: Atendimento e Gerenciamento de Agendamentos

Projeto acadêmico em grupo desenvolvido para a disciplina de APS, com orientação da professora Rita Cassiano.

O chatbot utiliza **Processamento de Linguagem Natural (NLP)** e **Machine Learning** para classificar mensagens em categorias e selecionar respostas. Para solicitações de agendamento, consulta, alteração e cancelamento, utiliza um banco de dados **SQLite**.

**Integrantes:** Luciene Aparecida Oliveira, Tatiane Avansi Caires e Valdeyr Moreira Nicacio.

> Os nomes, telefones, endereço, preços e agendamentos presentes nos exemplos são dados fictícios/demonstrativos. Este projeto é acadêmico e não representa um serviço comercial real.

## Tecnologias
- Python
- Pandas
- Scikit-learn (CountVectorizer e MultinomialNB)
- JSON
- SQLite

## Como executar
Execute as células em ordem. Se os arquivos CSV e JSON não estiverem no ambiente, a primeira etapa solicitará o upload deles no Google Colab.

## 1. Bibliotecas e arquivos de entrada

In [ ]:
import os
import json
import random
import sqlite3
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# No Google Colab, faça upload dos dois arquivos quando solicitado.
ARQUIVOS = ["treinamento_300_frases.csv", "respostas.json"]

faltantes = [arquivo for arquivo in ARQUIVOS if not os.path.exists(arquivo)]
if faltantes:
    try:
        from google.colab import files
        print("Envie os arquivos:", ", ".join(faltantes))
        enviados = files.upload()
        # Mantém os nomes esperados pelo projeto.
        for nome in faltantes:
            if nome not in enviados:
                candidatos = [n for n in enviados if n.lower().endswith(os.path.splitext(nome)[1])]
                if len(candidatos) == 1:
                    os.replace(candidatos[0], nome)
    except ImportError:
        raise FileNotFoundError(
            "Coloque treinamento_300_frases.csv e respostas.json na mesma pasta do notebook."
        )

for arquivo in ARQUIVOS:
    if not os.path.exists(arquivo):
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")

print("Arquivos localizados com sucesso!")

## 2. Carregamento e treinamento do modelo

In [ ]:
dados = pd.read_csv("treinamento_300_frases.csv")

# Verificações básicas da base de treinamento.
colunas_necessarias = {"pergunta", "categoria"}
if not colunas_necessarias.issubset(dados.columns):
    raise ValueError("O CSV precisa conter as colunas 'pergunta' e 'categoria'.")

dados = dados.dropna(subset=["pergunta", "categoria"]).copy()
dados["pergunta"] = dados["pergunta"].astype(str).str.strip()
dados["categoria"] = dados["categoria"].astype(str).str.strip()
dados = dados[(dados["pergunta"] != "") & (dados["categoria"] != "")]

with open("respostas.json", "r", encoding="utf-8") as arquivo:
    respostas = json.load(arquivo)

categorias_sem_resposta = set(dados["categoria"]) - set(respostas)
if categorias_sem_resposta:
    raise ValueError(f"Faltam respostas no JSON para: {sorted(categorias_sem_resposta)}")

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(dados["pergunta"])
y = dados["categoria"]

modelo = MultinomialNB()
modelo.fit(X, y)

print(f"Modelo treinado com {len(dados)} frases e {dados['categoria'].nunique()} categorias.")
display(dados["categoria"].value_counts().rename_axis("categoria").to_frame("quantidade"))

## 3. Função de classificação e resposta

In [ ]:
def responder(pergunta_usuario):
    """Classifica a mensagem e retorna uma resposta associada à categoria prevista."""
    pergunta_usuario = str(pergunta_usuario).strip()
    if not pergunta_usuario:
        return None, "Digite uma mensagem para começar."

    pergunta_vetorizada = vectorizer.transform([pergunta_usuario])
    categoria = modelo.predict(pergunta_vetorizada)[0]
    opcoes = respostas.get(categoria, ["Desculpe, não encontrei uma resposta para essa solicitação."])
    return categoria, random.choice(opcoes)

# Exemplos rápidos
for exemplo in ["Olá, bom dia!", "Qual o horário de funcionamento?", "Quanto custa o banho?"]:
    categoria, texto = responder(exemplo)
    print(f"Mensagem: {exemplo}\nCategoria: {categoria}\nBot: {texto}\n")

## 4. Banco de dados SQLite

In [ ]:
DB_PATH = "petshop.db"

def conectar():
    return sqlite3.connect(DB_PATH)

def inicializar_banco():
    with conectar() as conexao:
        cursor = conexao.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS clientes (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nome TEXT NOT NULL,
                telefone TEXT NOT NULL
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS pets (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nome_pet TEXT NOT NULL,
                raca TEXT,
                cliente_id INTEGER,
                FOREIGN KEY(cliente_id) REFERENCES clientes(id)
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS agendamentos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                pet_id INTEGER,
                servico TEXT,
                data TEXT,
                FOREIGN KEY(pet_id) REFERENCES pets(id)
            )
        """)

        # Insere exemplos apenas quando as tabelas estão vazias.
        cursor.execute("SELECT COUNT(*) FROM clientes")
        if cursor.fetchone()[0] == 0:
            cursor.executemany(
                "INSERT INTO clientes (nome, telefone) VALUES (?, ?)",
                [
                    ("Ana", "(11) 99999-1111"),
                    ("João", "(11) 99999-2222"),
                    ("Luciene", "(11) 99999-3333"),
                    ("Valdeyr", "(11) 99999-4444"),
                    ("Tatiane", "(11) 99999-5555"),
                ],
            )
            cursor.executemany(
                "INSERT INTO pets (nome_pet, raca, cliente_id) VALUES (?, ?, ?)",
                [
                    ("Rex", "Shih Tzu", 1),
                    ("Mel", "Poodle", 2),
                    ("Alex", "Doberman", 3),
                    ("Lulu", "Vira-lata", 4),
                    ("Kiara", "Beagle", 5),
                    ("Dolores", "Pastor Alemão", 5),
                ],
            )
            cursor.executemany(
                "INSERT INTO agendamentos (pet_id, servico, data) VALUES (?, ?, ?)",
                [(1, "Banho", "10/06/2026"), (2, "Tosa", "12/06/2026")],
            )
    print("Banco de dados pronto.")

inicializar_banco()

## 5. Operações de agendamento (CRUD)

In [ ]:
def create_agendamento(nome_pet, servico, data):
    with conectar() as conexao:
        cursor = conexao.cursor()
        cursor.execute("SELECT id FROM pets WHERE LOWER(nome_pet) = LOWER(?)", (nome_pet.strip(),))
        resultado = cursor.fetchone()
        if resultado is None:
            print(f"Pet '{nome_pet}' não encontrado. Cadastre-o antes de agendar.")
            return False
        cursor.execute(
            "INSERT INTO agendamentos (pet_id, servico, data) VALUES (?, ?, ?)",
            (resultado[0], servico, data),
        )
    print("Agendamento criado com sucesso!")
    return True

def consultar_agendamentos():
    with conectar() as conexao:
        dados_agendamentos = pd.read_sql_query("""
            SELECT agendamentos.id, pets.nome_pet, agendamentos.servico, agendamentos.data
            FROM agendamentos
            JOIN pets ON pets.id = agendamentos.pet_id
            ORDER BY agendamentos.id
        """, conexao)
    return dados_agendamentos

def alterar_agendamento(id_agendamento, nova_data):
    with conectar() as conexao:
        cursor = conexao.cursor()
        cursor.execute(
            "UPDATE agendamentos SET data = ? WHERE id = ?",
            (nova_data, int(id_agendamento)),
        )
        alterados = cursor.rowcount
    print("Agendamento alterado." if alterados else "Agendamento não encontrado.")
    return bool(alterados)

def cancelar_agendamento(id_agendamento):
    with conectar() as conexao:
        cursor = conexao.cursor()
        cursor.execute("DELETE FROM agendamentos WHERE id = ?", (int(id_agendamento),))
        removidos = cursor.rowcount
    print("Agendamento cancelado." if removidos else "Agendamento não encontrado.")
    return bool(removidos)

display(consultar_agendamentos())

## 6. Conversa com o chatbot

In [ ]:
def conversar():
    """Inicia uma conversa interativa. Digite 'sair' para encerrar."""
    print("Chatbot PetShop iniciado! Digite 'sair' para encerrar.\n")
    while True:
        pergunta_usuario = input("Você: ").strip()
        if pergunta_usuario.lower() in {"sair", "tchau", "encerrar"}:
            print("Bot: Obrigado pelo contato! Até logo.")
            break

        categoria, texto = responder(pergunta_usuario)
        print(f"Bot: {texto}")

        if categoria == "agendar":
            nome_pet = input("Nome do pet: ").strip()
            servico = input("Serviço desejado: ").strip()
            data = input("Data desejada (DD/MM/AAAA): ").strip()
            create_agendamento(nome_pet, servico, data)
        elif categoria == "consultar_agendamento":
            display(consultar_agendamentos())
        elif categoria == "alterar_agendamento":
            try:
                id_agendamento = int(input("ID do agendamento: "))
                nova_data = input("Nova data (DD/MM/AAAA): ").strip()
                alterar_agendamento(id_agendamento, nova_data)
            except ValueError:
                print("Informe um ID numérico válido.")
        elif categoria == "cancelar_agendamento":
            try:
                id_agendamento = int(input("ID do agendamento: "))
                cancelar_agendamento(id_agendamento)
            except ValueError:
                print("Informe um ID numérico válido.")

# Para iniciar a conversa, execute:
# conversar()

## 7. Testes manuais

Abaixo estão exemplos para testar as funções sem iniciar o diálogo interativo. Os exemplos de criação/alteração/cancelamento modificam o banco local.

```python
categoria, resposta = responder("Quero marcar um banho para meu cachorro")
print(categoria, resposta)

create_agendamento("Rex", "Banho", "20/06/2026")
display(consultar_agendamentos())
```

Para iniciar a conversa completa, remova o comentário da chamada `conversar()` na última célula de código.

## Créditos e contexto

Trabalho acadêmico em grupo desenvolvido para a disciplina de **Atividades Práticas Supervisionadas (APS)**, com orientação da **Professora Rita Cassiano**.

**Integrantes:**
- Luciene Aparecida Oliveira
- Tatiane Avansi Caires
- Valdeyr Moreira Nicacio

O notebook foi organizado para facilitar a execução e a apresentação do trabalho. As funcionalidades e a proposta permanecem baseadas no projeto acadêmico original.